# Modelo e avaliação — EEGNet no BCI IV 2a

Predições do EEGNet (Lawhern et al., 2018) com `GroupKFold` por sujeito, 9 folds
([`../src/train.py`](../src/train.py)). Síntese na seção 13 do [`../README.md`](../README.md).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.metrics import (
    ConfusionMatrixDisplay, accuracy_score, balanced_accuracy_score, confusion_matrix,
    f1_score, matthews_corrcoef,
)

data_dir = Path("..") / "data"
pred = pd.read_parquet(data_dir / "predictions" / "predictions_eegnet.parquet")
subjects = pd.read_parquet(data_dir / "processed" / "dim_subject.parquet")
CLASSES = ["left_hand", "right_hand", "feet", "tongue"]
pred["correct"] = pred["y_true"] == pred["y_pred"]
clean = pred[~pred["has_artifact"]]
print(pred["model_version"].unique(), len(pred), "predições,", pred["subject_id"].nunique(), "sujeitos")
pred.head()

## Divisão e prevalência

Sujeito de teste de cada fold e prevalência das classes (acaso = 25%).

In [ ]:
pred.groupby("fold")["subject_id"].unique().rename("sujeito de teste").to_frame()

In [ ]:
pd.DataFrame({
    "todos (%)": pred["y_true"].map(dict(enumerate(CLASSES))).value_counts(normalize=True) * 100,
    "sem artefato (%)": clean["y_true"].map(dict(enumerate(CLASSES))).value_counts(normalize=True) * 100,
}).round(1)

## Métricas com IC 95%

Trials sem artefato. IC por bootstrap reamostrando sujeitos (2.000 réplicas).

In [ ]:
METRICS = {
    "acurácia": accuracy_score,
    "acurácia balanceada": balanced_accuracy_score,
    "F1 macro": lambda t, p: f1_score(t, p, average="macro"),
    "MCC": matthews_corrcoef,
}


def metrics_with_ci(df, n_boot=2000, seed=42):
    rng = np.random.default_rng(seed)
    by_subject = {s: g[["y_true", "y_pred"]].to_numpy() for s, g in df.groupby("subject_id")}
    ids = list(by_subject)
    reps = []
    for _ in range(n_boot):
        sample = np.concatenate([by_subject[s] for s in rng.choice(ids, len(ids))])
        reps.append([f(sample[:, 0], sample[:, 1]) for f in METRICS.values()])
    low, high = np.percentile(reps, [2.5, 97.5], axis=0)
    point = [f(df["y_true"], df["y_pred"]) for f in METRICS.values()]
    return pd.DataFrame({"valor": point, "IC 2,5%": low, "IC 97,5%": high}, index=list(METRICS))


summary = pd.concat({"sem artefato": metrics_with_ci(clean), "todos": metrics_with_ci(pred)}, axis=1)
summary.round(3)

## Sensibilidade e especificidade por classe

Uma classe contra as demais, com IC 95% de Wilson.

In [ ]:
def wilson(k, n):
    ci = stats.binomtest(int(k), int(n)).proportion_ci(method="wilson")
    return k / n, ci.low, ci.high


rows = []
for k, name in enumerate(CLASSES):
    pos, neg = clean[clean["y_true"] == k], clean[clean["y_true"] != k]
    sens = wilson((pos["y_pred"] == k).sum(), len(pos))
    spec = wilson((neg["y_pred"] != k).sum(), len(neg))
    rows.append({"classe": name, "sensibilidade": sens[0], "sens IC": f"{sens[1]:.2f}–{sens[2]:.2f}",
                 "especificidade": spec[0], "espec IC": f"{spec[1]:.2f}–{spec[2]:.2f}"})
pd.DataFrame(rows).round(3)

## Matriz de confusão

Linhas = classe verdadeira; à direita, normalizada por linha.

In [ ]:
cm = confusion_matrix(clean["y_true"], clean["y_pred"])
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(ax=axes[0], colorbar=False, cmap="Blues")
ConfusionMatrixDisplay(cm / cm.sum(1, keepdims=True), display_labels=CLASSES).plot(
    ax=axes[1], colorbar=False, cmap="Blues", values_format=".2f")
axes[0].set_title("contagens"); axes[1].set_title("normalizada por linha")
plt.tight_layout()

In [ ]:
off = [(CLASSES[i], CLASSES[j], cm[i, j]) for i in range(4) for j in range(4) if i != j]
pd.DataFrame(off, columns=["verdadeira", "predita", "trials"]).sort_values("trials", ascending=False).head(6)

## Acurácia por sujeito

IC 95% de Wilson, trials sem artefato.

In [ ]:
per_subject = clean.groupby("subject_id").agg(
    fold=("fold", "first"), n=("correct", "size"), acertos=("correct", "sum"))
ci = per_subject.apply(lambda r: wilson(r["acertos"], r["n"]), axis=1, result_type="expand")
per_subject[["acurácia", "IC 2,5%", "IC 97,5%"]] = ci
per_subject = per_subject.join(subjects.set_index("subject_id")[["age", "sex"]])
per_subject.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
err = [per_subject["acurácia"] - per_subject["IC 2,5%"], per_subject["IC 97,5%"] - per_subject["acurácia"]]
ax.errorbar(per_subject.index, per_subject["acurácia"], yerr=err, fmt="o", capsize=4)
ax.axhline(0.25, ls="--", c="gray", label="acaso (25%)")
ax.set_ylabel("acurácia (sem artefato)"); ax.set_ylim(0, 1); ax.legend()
plt.tight_layout()

## Sessão e artefato

Qui-quadrado (Fisher se alguma frequência esperada for < 5), com correção de Holm.

In [ ]:
def compare(df, column):
    table = pd.crosstab(df[column], df["correct"]).to_numpy()
    if (stats.contingency.expected_freq(table) < 5).any():
        return stats.fisher_exact(table)[1], "Fisher"
    return stats.chi2_contingency(table)[1], "qui-quadrado"


def holm(pvalues):
    order = np.argsort(pvalues)
    adjusted = np.empty(len(pvalues))
    running = 0.0
    for rank, i in enumerate(order):
        running = max(running, min(1.0, pvalues[i] * (len(pvalues) - rank)))
        adjusted[i] = running
    return adjusted


groups = {"session_id": clean, "has_artifact": pred}
tests = {column: compare(df, column) for column, df in groups.items()}
p_holm = holm([p for p, _ in tests.values()])
for (column, (p, name)), p_adj in zip(tests.items(), p_holm):
    print(f"{column}: {name}, p = {p:.3g}, p (Holm) = {p_adj:.3g}")
    display(groups[column].groupby(column)["correct"].agg(n="size", acurácia="mean").round(3))

## Limitações

- 9 sujeitos: IC largo, e o bootstrap com 9 grupos é aproximado.
- Domain shift entre pessoas: o teste é sempre um sujeito não visto, então os números não se
  comparam com treino e teste no mesmo sujeito.
- Época escolhida em um único sujeito de validação; hiperparâmetros do artigo, sem busca.
- A saída do modelo não é probabilidade calibrada (0,8 não é 80% de chance).
- No uso real não há marcação de artefato.
- Voluntários saudáveis em laboratório.